# 24 · Stack of every model family (E54)

All members out of fold on the same 5 brand-grouped folds (from notebook 23): CatBoost final model, + age of the check, + e5 embeddings, TabPFN v2, TabICL, fine-tuned row as text, CatBoost + fine-tuned text correction. The stack learns non-negative weights summing to 1 per platform, minimising MAE, on 4 folds and is applied to the 5th.

In [1]:
import sys, json, pickle, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 220, "display.max_columns", 30)
import extra_models as em, history as H, stack_all as sa

In [2]:
f, meta = em.load(ROOT); store = pickle.load(open(ROOT / "data" / "processed" / "finetune_oof_cache.pkl", "rb"))
M = sa.members(f, meta, ROOT, store, ROOT / "data" / "processed" / "embeddings_e5_cache.pkl")
M.groupby(meta.platform.values).apply(lambda q: (q.sub(meta.y.values[q.index], axis=0)).abs().mean()).T.round(3)

,tiktok,twitter,youtube
CatBoost (final model),0.271,0.467,0.453
CatBoost + age of check,0.272,0.452,0.449
CatBoost + embeddings,0.269,0.458,0.461
TabPFN v2,0.268,0.466,0.443
TabICL,0.272,0.466,0.445
MLP (ENG + embeddings),0.280,0.465,0.462
fine-tuned row as text,0.289,0.490,0.454
CatBoost + fine-tuned text correction,0.275,0.464,0.444


In [3]:
st, W = sa.stack(M, meta, store["fid"])
best3 = {p: M[meta.platform.values == p].sub(meta.y.values[meta.platform.values == p], axis=0).abs().mean().nsmallest(3).index.tolist() for p in sa.PLATS}
b3 = np.array([M.loc[i, best3[p]].mean() for i, p in enumerate(meta.platform.values)])
P = sa.long(M, meta, {"mean of all members": M.mean(axis=1).values, "mean of the best three": b3, "stack (nested weights)": st})
S = H.summarise(P); S.pivot(index="model", columns="platform", values="MAE")[sa.PLATS].round(3)

platform,tiktok,youtube,twitter
model,,,
CatBoost (final model),0.271,0.453,0.467
CatBoost + age of check,0.272,0.449,0.452
CatBoost + embeddings,0.269,0.461,0.458
CatBoost + fine-tuned text correction,0.275,0.444,0.464
MLP (ENG + embeddings),0.280,0.462,0.465
TabICL,0.272,0.445,0.466
TabPFN v2,0.268,0.443,0.466
fine-tuned row as text,0.289,0.454,0.490
mean of all members,0.262,0.430,0.447


In [4]:
W.groupby("platform").mean().drop(columns="fold").T.round(2)

platform,tiktok,twitter,youtube
CatBoost (final model),0.00,0.00,0.00
CatBoost + age of check,0.00,0.51,0.00
CatBoost + embeddings,0.04,0.00,0.00
TabPFN v2,0.42,0.00,0.08
TabICL,0.02,0.00,0.10
MLP (ENG + embeddings),0.10,0.35,0.23
fine-tuned row as text,0.04,0.02,0.32
CatBoost + fine-tuned text correction,0.38,0.12,0.26


In [5]:
B = "CatBoost (final model)"
best = {p: S[(S.platform == p) & ~S.model.str.startswith(("mean", "stack"))].nsmallest(1, "MAE").model.iloc[0] for p in sa.PLATS}
G = pd.DataFrame([{"platform": p, "base": b, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P, p, b, k)))} for p in sa.PLATS for b in dict.fromkeys([B, best[p]]) for k in P.model.unique() if k != b]); G.round(3)

,platform,base,model,gain,CI low,CI high
0,tiktok,CatBoost (final model),CatBoost + age of check,-0.001,-0.003,0.001
1,tiktok,CatBoost (final model),CatBoost + embeddings,0.001,-0.002,0.005
2,tiktok,CatBoost (final model),TabPFN v2,0.003,0.000,0.005
3,tiktok,CatBoost (final model),TabICL,-0.001,-0.005,0.003
4,tiktok,CatBoost (final model),MLP (ENG + embeddings),-0.010,-0.015,-0.004
5,tiktok,CatBoost (final model),fine-tuned row as text,-0.018,-0.025,-0.012
6,tiktok,CatBoost (final model),CatBoost + fine-tuned text correction,-0.005,-0.011,0.002
7,tiktok,CatBoost (final model),mean of all members,0.009,0.007,0.011
8,tiktok,CatBoost (final model),mean of the best three,0.005,0.003,0.006
9,tiktok,CatBoost (final model),stack (nested weights),0.011,0.008,0.014


## Takeaways
See `docs/en/experiments.md` (E54) and `agent/vault/E54.md`. Numbers: `data/processed/stack_all_numbers.json`.

In [6]:
json.dump({"models": S.to_dict("records"), "weights": W.to_dict("records"), "gains": G.to_dict("records"), "best_single": best, "best_three": best3}, open(ROOT / "data" / "processed" / "stack_all_numbers.json", "w"), indent=1, default=float)